# <font color='blue'>Tarea 5. </font>
### <font color='red'>12 de enero de 2026</font>
### Andrés Padrón Quintana 
### <font color='green'>Módulo 5. DATA SCIENCE AND MACHINE LEARNING APPLIED TO FINANCIAL MARKETS </font>

Intentar el ejercicio de arriba y reportar el accuracy para epoch 4000 de los mejores modelos de abajo
1. Norma 1 (buscar una lambda entre `values=[1e-1,1e-2,1e-3,1e-4,1e-5,1e-6]` ) y qudarse con el mejor en términos de accuracy

2. Norma 1 y 2 (Redes Elásticas) `values=[1e-1,1e-2,1e-3,1e-4,1e-5,1e-6]`
3. Regularizando el Bias `values=[1e-1,1e-2,1e-3,1e-4,1e-5,1e-6]`
4. Combinando:  
    1. a) 1 y 3 `values=[1e-1,1e-2,1e-3,1e-4,1e-5,1e-6]`    
    2. b) 2 y 3. `values=[1e-1,1e-2,1e-3,1e-4,1e-5,1e-6]`

¿Qué modelo tuvo el mejor accuracy en test?

Usar el split de arriba `seed=1` y 30% Train y 70% Test. 


In [2]:
# Importamos paquetes
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.datasets import make_moons
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.regularizers import l1, l2, l1_l2
from tensorflow.keras import backend as K

# Reproducibilidad
np.random.seed(1)
tf.random.set_seed(1)


2026-01-10 18:20:30.965623: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
# Igual que el notebook
X, y = make_moons(n_samples=100, noise=0.2, random_state=1)

# Split: 30% train (n_train=30) y 70% test
n_train = 30
trainX, testX = X[:n_train, :], X[n_train:, :]
trainy, testy = y[:n_train], y[n_train:]


In [4]:
def build_model(reg_kernel=None, reg_bias=None):
    """
    Arquitectura igual al notebook:
    Dense(500, relu) -> Dense(1, sigmoid)
    Regularización aplicada SOLO a la primera capa (como se hace típicamente).
    """
    model = Sequential()
    model.add(Dense(
        500, input_dim=2, activation="relu",
        kernel_regularizer=reg_kernel,
        bias_regularizer=reg_bias
    ))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model

def fit_eval(model, trainX, trainy, testX, testy, epochs=4000):
    """
    Entrena y regresa accuracy en train y test al final (epoch 4000).
    """
    model.fit(trainX, trainy, epochs=epochs, verbose=0)
    _, train_acc = model.evaluate(trainX, trainy, verbose=0)
    _, test_acc  = model.evaluate(testX,  testy,  verbose=0)
    return float(train_acc), float(test_acc)


In [5]:
values = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
rows = []

def run_family(nombre, make_regs):
    """
    nombre: string de la familia
    make_regs(lam) -> (reg_kernel, reg_bias)
    """
    for lam in values:
        K.clear_session()  # importante para no acumular grafos entre corridas
        reg_k, reg_b = make_regs(lam)
        model = build_model(reg_kernel=reg_k, reg_bias=reg_b)
        tr, te = fit_eval(model, trainX, trainy, testX, testy, epochs=4000)
        rows.append({
            "familia": nombre,
            "lambda": lam,
            "train_acc": tr,
            "test_acc": te
        })

# 1) Norma 1 (L1 en kernel)
run_family("1) L1 kernel", lambda lam: (l1(lam), None))

# 2) Norma 1 y 2 (Elastic Net: L1+L2 en kernel, usando l1=l2=lambda)
run_family("2) ElasticNet kernel (l1=l2)", lambda lam: (l1_l2(l1=lam, l2=lam), None))

# 3) Regularizando el Bias (L1 en bias)
run_family("3) L1 bias", lambda lam: (None, l1(lam)))

# 4A) Combinando 1 y 3 (L1 kernel + L1 bias)
run_family("4A) L1 kernel + L1 bias", lambda lam: (l1(lam), l1(lam)))

# 4B) Combinando 2 y 3 (ElasticNet kernel + L1 bias)
run_family("4B) ElasticNet kernel + L1 bias", lambda lam: (l1_l2(l1=lam, l2=lam), l1(lam)))

df = pd.DataFrame(rows)
df.head()


/Users/andrespadronquintana/anaconda3/lib/python3.11/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


,familia,lambda,train_acc,test_acc
0,1) L1 kernel,0.10000,0.900000,0.814286
1,1) L1 kernel,0.01000,0.933333,0.828571
2,1) L1 kernel,0.00100,1.000000,0.942857
3,1) L1 kernel,0.00010,1.000000,0.928571
4,1) L1 kernel,0.00001,1.000000,0.914286


In [6]:
# Mejor por familia (criterio: mayor test_acc; desempate por train_acc)
best_by_family = (
    df.sort_values(["familia", "test_acc", "train_acc"], ascending=[True, False, False])
      .groupby("familia", as_index=False)
      .head(1)
      .sort_values("test_acc", ascending=False)
      .reset_index(drop=True)
)

best_by_family


,familia,lambda,train_acc,test_acc
0,2) ElasticNet kernel (l1=l2),0.001,1.0,0.957143
1,1) L1 kernel,0.001,1.0,0.942857
2,4A) L1 kernel + L1 bias,0.001,1.0,0.942857
3,4B) ElasticNet kernel + L1 bias,0.001,1.0,0.942857
4,3) L1 bias,0.001,1.0,0.928571


El mejor modelo fue:

2) ElasticNet kernel (l1=l2) con λ = 0.001, con test accuracy = 0.957143 (y train accuracy = 1.0).

In [7]:
best_global = best_by_family.iloc[0]

print("MEJOR POR FAMILIA (ordenado por test_acc):\n")
print(best_by_family[["familia", "lambda", "train_acc", "test_acc"]].to_string(index=False))

print("\n" + "="*70)
print("MEJOR GLOBAL EN TEST:")
print(f"Familia  : {best_global['familia']}")
print(f"Lambda   : {best_global['lambda']}")
print(f"Train_acc: {best_global['train_acc']:.6f}")
print(f"Test_acc : {best_global['test_acc']:.6f}")


MEJOR POR FAMILIA (ordenado por test_acc):

                        familia  lambda  train_acc  test_acc
   2) ElasticNet kernel (l1=l2)   0.001        1.0  0.957143
                   1) L1 kernel   0.001        1.0  0.942857
        4A) L1 kernel + L1 bias   0.001        1.0  0.942857
4B) ElasticNet kernel + L1 bias   0.001        1.0  0.942857
                     3) L1 bias   0.001        1.0  0.928571

MEJOR GLOBAL EN TEST:
Familia  : 2) ElasticNet kernel (l1=l2)
Lambda   : 0.001
Train_acc: 1.000000
Test_acc : 0.957143


Usando `seed = 1` con split **30% train** y **70% test**, se evaluaron 5 familias de regularización para

$$
\lambda \in \left\{10^{-1},\,10^{-2},\,10^{-3},\,10^{-4},\,10^{-5},\,10^{-6}\right\}.
$$



**Conclusión.** El mejor desempeño en test se obtuvo con **ElasticNet en los pesos** \((l_1=l_2)\) con \(\lambda = 0.01\), alcanzando:

$$
\text{Test acc} = 0.9571.
$$